In [15]:
import torch
import torch.nn.functional as F
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load dataset
df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")      # change if required

# Load models
deberta_name = "microsoft/deberta-v3-small"
roberta_name = "roberta-base"

deberta_tokenizer = AutoTokenizer.from_pretrained(deberta_name)
roberta_tokenizer = AutoTokenizer.from_pretrained(roberta_name)

deberta_model = AutoModelForSequenceClassification.from_pretrained(deberta_name).to(device)
roberta_model = AutoModelForSequenceClassification.from_pretrained(roberta_name).to(device)

deberta_model.eval()
roberta_model.eval()

labels = ["A","B","C","D","E"]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias         

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
def option_probability(model, tokenizer, prompt, option):

    encoding = tokenizer(
        prompt,
        option,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        logits = model(**encoding).logits

    probs = F.softmax(logits, dim=-1)

    # probability that this option is correct
    return probs[0][1].item()

In [17]:
row = df.loc[25]

prompt = row["prompt"]

options = {
    "A": row["A"],
    "B": row["B"],
    "C": row["C"],
    "D": row["D"],
    "E": row["E"]
}

## Q1

In [18]:
deberta_scores = {}

for key, value in options.items():
    deberta_scores[key] = option_probability(
        deberta_model,
        deberta_tokenizer,
        prompt,
        value
    )

best_option = max(deberta_scores, key=deberta_scores.get)

print(best_option)
print(deberta_scores[best_option])

D
0.46630859375


## Q2

In [19]:
roberta_scores = {}

for key, value in options.items():
    roberta_scores[key] = option_probability(
        roberta_model,
        roberta_tokenizer,
        prompt,
        value
    )

avg_scores = {}

for k in labels:
    avg_scores[k] = (
        deberta_scores[k] +
        roberta_scores[k]
    ) / 2

best_option = max(avg_scores, key=avg_scores.get)

print(best_option)
print(avg_scores[best_option])

D
0.4706868976354599


## Q3

In [20]:
weighted_scores = {}

for k in labels:
    weighted_scores[k] = (
        0.7 * deberta_scores[k]
        +
        0.3 * roberta_scores[k]
    )

best_option = max(weighted_scores, key=weighted_scores.get)

print(best_option)
print(weighted_scores[best_option])

D
0.4689355760812759


## Q4

In [21]:
ranking = sorted(
    weighted_scores.items(),
    key=lambda x: x[1],
    reverse=True
)

top3 = " ".join(
    option
    for option, score in ranking[:3]
)

print(top3)

D E B


## Q5

In [ ]:
predictions = []

for _, row in df.iterrows():

    prompt = row["prompt"]

    options = {
        "A": row["A"],
        "B": row["B"],
        "C": row["C"],
        "D": row["D"],
        "E": row["E"],
    }

    deberta_scores = {}
    roberta_scores = {}

    for label, option in options.items():

        deberta_scores[label] = option_probability(
            deberta_model,
            deberta_tokenizer,
            prompt,
            option,
        )

        roberta_scores[label] = option_probability(
            roberta_model,
            roberta_tokenizer,
            prompt,
            option,
        )

    weighted_scores = {
        label: 0.7 * deberta_scores[label] + 0.3 * roberta_scores[label]
        for label in labels
    }

    top3 = sorted(
        weighted_scores,
        key=weighted_scores.get,
        reverse=True,
    )[:3]

    predictions.append(" ".join(top3))

submission = pd.DataFrame({
    "id": df["id"],
    "prediction": predictions,
})

submission.to_csv("submission.csv", index=False)

print(submission.head())
print(f"\nNumber of prediction rows (excluding header): {len(submission)}")

   id prediction
0   1      C D A
1   2      B D A
2   3      D A C
3   4      D B E
4   5      C A D

Number of prediction rows (excluding header): 500


## Q6

In [ ]:
instruction = "Answer the following multiple-choice question carefully:"

changed_predictions = 0

for _, row in df.head(50).iterrows():

    prompt = row["prompt"]

    augmented_prompt = instruction + "\n\n" + prompt

    options = {
        "A": row["A"],
        "B": row["B"],
        "C": row["C"],
        "D": row["D"],
        "E": row["E"],
    }

    # Original prompt scores
    original_scores = {}
    for label, option in options.items():
        original_scores[label] = option_probability(
            deberta_model,
            deberta_tokenizer,
            prompt,
            option,
        )

    original_top1 = max(original_scores, key=original_scores.get)

    # TTA scores (average original + augmented)
    tta_scores = {}
    for label, option in options.items():

        p_original = option_probability(
            deberta_model,
            deberta_tokenizer,
            prompt,
            option,
        )

        p_augmented = option_probability(
            deberta_model,
            deberta_tokenizer,
            augmented_prompt,
            option,
        )

        tta_scores[label] = (p_original + p_augmented) / 2

    tta_top1 = max(tta_scores, key=tta_scores.get)

    if original_top1 != tta_top1:
        changed_predictions += 1

print("Number of changed Top-1 predictions:", changed_predictions)

Number of changed Top-1 predictions: 12


## Q7

In [ ]:
different_predictions = 0

for _, row in df.head(100).iterrows():

    prompt = row["prompt"]

    options = {
        "A": row["A"],
        "B": row["B"],
        "C": row["C"],
        "D": row["D"],
        "E": row["E"],
    }

    deberta_scores = {}
    roberta_scores = {}

    # Get probabilities for each option
    for label, option in options.items():

        deberta_scores[label] = option_probability(
            deberta_model,
            deberta_tokenizer,
            prompt,
            option,
        )

        roberta_scores[label] = option_probability(
            roberta_model,
            roberta_tokenizer,
            prompt,
            option,
        )

    # DeBERTa Top-1
    deberta_top1 = max(deberta_scores, key=deberta_scores.get)

    # Weighted Ensemble (0.7 DeBERTa + 0.3 RoBERTa)
    weighted_scores = {
        label: 0.7 * deberta_scores[label] + 0.3 * roberta_scores[label]
        for label in labels
    }

    weighted_top1 = max(weighted_scores, key=weighted_scores.get)

    if deberta_top1 != weighted_top1:
        different_predictions += 1

print("Number of rows with different Top-1 predictions:", different_predictions)

Number of rows with different Top-1 predictions: 4


## Q8

In [ ]:
positive_confidence_gain = 0

for _, row in df.head(100).iterrows():

    prompt = row["prompt"]

    options = {
        "A": row["A"],
        "B": row["B"],
        "C": row["C"],
        "D": row["D"],
        "E": row["E"],
    }

    deberta_scores = {}
    roberta_scores = {}

    for label, option in options.items():

        deberta_scores[label] = option_probability(
            deberta_model,
            deberta_tokenizer,
            prompt,
            option,
        )

        roberta_scores[label] = option_probability(
            roberta_model,
            roberta_tokenizer,
            prompt,
            option,
        )

    deberta_confidence = max(deberta_scores.values())

    weighted_scores = {
        label: 0.7 * deberta_scores[label] + 0.3 * roberta_scores[label]
        for label in labels
    }

    ensemble_confidence = max(weighted_scores.values())

    confidence_gain = ensemble_confidence - deberta_confidence

    if confidence_gain > 0:
        positive_confidence_gain += 1

print("Positive confidence gain:", positive_confidence_gain)

Positive confidence gain: 36


## Q9

In [ ]:
top3_changes = 0

for _, row in df.head(100).iterrows():

    prompt = row["prompt"]

    options = {
        "A": row["A"],
        "B": row["B"],
        "C": row["C"],
        "D": row["D"],
        "E": row["E"],
    }

    deberta_scores = {}
    roberta_scores = {}

    for label, option in options.items():

        deberta_scores[label] = option_probability(
            deberta_model,
            deberta_tokenizer,
            prompt,
            option,
        )

        roberta_scores[label] = option_probability(
            roberta_model,
            roberta_tokenizer,
            prompt,
            option,
        )

    deberta_top3 = sorted(
        deberta_scores,
        key=deberta_scores.get,
        reverse=True,
    )[:3]

    weighted_scores = {
        label: 0.7 * deberta_scores[label] + 0.3 * roberta_scores[label]
        for label in labels
    }

    ensemble_top3 = sorted(
        weighted_scores,
        key=weighted_scores.get,
        reverse=True,
    )[:3]

    if deberta_top3 != ensemble_top3:
        top3_changes += 1

print("Rows with Top-3 ranking change:", top3_changes)

Rows with Top-3 ranking change: 9


## Q10

In [ ]:
train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

def map_at_3(actual, predicted):
    score = 0.0

    for a, p in zip(actual, predicted):
        if a in p[:3]:
            score += 1.0 / (p.index(a) + 1)

    return score / len(actual)

predictions = []
ground_truth = []

for _, row in train_df.head(100).iterrows():

    prompt = row["prompt"]

    options = {
        "A": row["A"],
        "B": row["B"],
        "C": row["C"],
        "D": row["D"],
        "E": row["E"],
    }

    deberta_scores = {}
    roberta_scores = {}

    for label, option in options.items():

        deberta_scores[label] = option_probability(
            deberta_model,
            deberta_tokenizer,
            prompt,
            option
        )

        roberta_scores[label] = option_probability(
            roberta_model,
            roberta_tokenizer,
            prompt,
            option
        )

    weighted_scores = {
        label: 0.7 * deberta_scores[label] + 0.3 * roberta_scores[label]
        for label in labels
    }

    top3 = sorted(
        weighted_scores,
        key=weighted_scores.get,
        reverse=True
    )[:3]

    predictions.append(top3)
    ground_truth.append(row["answer"])   # Change if your answer column has another name

score = map_at_3(ground_truth, predictions)

print("MAP@3:", round(score, 4))

MAP@3: 0.4
